# Orquestrador Automático de Tutoria (PBL)
Este notebook extrai sumários de livros de medicina em PDF do Google Drive, mapeia objetivos usando a inteligência do Gemini (Google GenAI SDK), realiza curadoria de vídeos didáticos no YouTube (PT-BR) e gera PDFs consolidados com capas premium.

In [ ]:
# 1. Instalação de Dependências
!pip install pymupdf pypdf reportlab pydantic python-dotenv google-genai httpx --quiet


In [ ]:
# 2. Montagem do Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3. Configuração das APIs (Secrets)
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['YOUTUBE_API_KEY'] = userdata.get('YOUTUBE_API_KEY')
print('Chaves configuradas com sucesso!')


In [ ]:
# === DIAGNÓSTICO: Listar Modelos Disponíveis ===
from google import genai
import os

diag_client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))
print('=== MODELOS GEMINI FLASH DISPONÍVEIS ===')
for model in diag_client.models.list():
    name = model.name if hasattr(model, 'name') else str(model)
    display = model.display_name if hasattr(model, 'display_name') else ''
    if 'flash' in name.lower():
        print(f'  ✅ {name}  ({display})')
print('=========================================')


In [ ]:
# 4. Motor do Orquestrador
import asyncio
import logging
import json
import os
import io
import random
import urllib.parse
import httpx
import pydantic
import fitz
from google import genai
from google.genai import types
from pypdf import PdfReader, PdfWriter, PageObject
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import inch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("pypdf").setLevel(logging.ERROR)

import typing
class Corte(pydantic.BaseModel):
    arquivo: str
    capitulo: str
    nivel: typing.Literal['conceito', 'mecanismo', 'clinica']
    pagina_inicial: int
    pagina_final: int

class VideoSugerido(pydantic.BaseModel):
    termo_busca: str
    video_id: str
    titulo_formatado: str

class PlanejamentoVideos(pydantic.BaseModel):
    termos_busca: list[str]

class CuradoriaVideo(pydantic.BaseModel):
    video_escolhido_id: str
    titulo_formatado: str

class Objetivo(pydantic.BaseModel):
    numero: str
    titulo: str
    cortes: list[Corte]
    videos: list[VideoSugerido] = []

class RoteiroTutoria(pydantic.BaseModel):
    objetivos: list[Objetivo]

class RefinamentoCorte(pydantic.BaseModel):
    relevante: bool
    pagina_inicial_refinada: int
    pagina_final_refinada: int

class TocItem(pydantic.BaseModel):
    nivel: int
    titulo: str
    pagina: int

class TocExtracted(pydantic.BaseModel):
    itens: list[TocItem]

def draw_wrapped_text(c, text, width, x, y, font, size, line_height, color):
    c.setFont(font, size)
    c.setFillColor(color)
    words = text.split(' ')
    lines = []
    current_line = []
    for word in words:
        current_line.append(word)
        if c.stringWidth(' '.join(current_line), font, size) > width:
            current_line.pop()
            lines.append(' '.join(current_line))
            current_line = [word]
    if current_line:
        lines.append(' '.join(current_line))
    for line in lines:
        c.drawString(x, y, line)
        y -= line_height
    return y

def create_cover_page(objetivo_numero, objetivo_titulo, videos=None):
    if videos is None: videos = []
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=A4)
    width, height = A4
    c.setFillColorRGB(1, 1, 1)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    c.setFillColorRGB(0.12, 0.53, 0.90)
    c.rect(0, 0, 15, height, fill=1, stroke=0)
    c.setFillColorRGB(0.1, 0.1, 0.1)
    c.setFont("Helvetica-Bold", 46)
    c.drawString(60, height - 120, f"OBJETIVO {objetivo_numero}")
    c.setStrokeColorRGB(0.8, 0.8, 0.8)
    c.setLineWidth(1)
    c.line(60, height - 140, width - 60, height - 140)
    c.setFont("Helvetica", 16)
    c.setFillColorRGB(0.4, 0.4, 0.4)
    c.drawString(60, height - 170, "ROTEIRO DE TUTORIA")
    font_size = 22
    line_height = 30
    if len(objetivo_titulo) > 150:
        font_size = 18
        line_height = 24
    if len(objetivo_titulo) > 250:
        font_size = 14
        line_height = 18
    end_y = draw_wrapped_text(c, objetivo_titulo, width - 120, 60, height - 230, "Helvetica-Bold", font_size, line_height, colors.black)
    if videos:
        vy = min(end_y - 40, height - 300)
        c.setFont("Helvetica-Bold", 14)
        c.setFillColorRGB(0.12, 0.53, 0.90)
        c.drawString(60, vy, "Vídeos Sugeridos:")
        vy -= 25
        for vid in videos[:5]:
            if vy < 70: break
            url = f"https://youtube.com/watch?v={vid.video_id}"
            c.setFont("Helvetica", 11)
            c.setFillColorRGB(0.3, 0.3, 0.3)
            c.drawString(60, vy, "•")
            c.setFont("Helvetica-BoldOblique", 11)
            c.setFillColorRGB(0.06, 0.45, 0.85)
            c.drawString(75, vy, f"{vid.titulo_formatado}")
            text_width = c.stringWidth(f"{vid.titulo_formatado}", "Helvetica-BoldOblique", 11)
            line_end_x = min(530, 75 + text_width)
            c.setStrokeColorRGB(0.06, 0.45, 0.85)
            c.setLineWidth(0.6)
            c.line(75, vy - 2, line_end_x, vy - 2)
            c.linkURL(url, (75, vy - 2, line_end_x, vy + 10), relative=0)
            vy -= 24
    c.setFont("Helvetica-Oblique", 10)
    c.setFillColorRGB(0.6, 0.6, 0.6)
    c.drawString(60, 40, "© Conteúdo Autoral • João Gabriel R. Trovão")
    c.save()
    packet.seek(0)
    return PdfReader(packet).pages[0]

def create_separator_page(livro, capitulo):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=A4)
    width, height = A4
    c.setFillColorRGB(1, 1, 1)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    c.setFillColorRGB(0.12, 0.53, 0.90)
    c.rect(40, height/2 + 20, width - 80, 60, fill=1, stroke=0)
    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 24)
    c.drawCentredString(width/2, height/2 + 40, "REFERÊNCIA")
    draw_wrapped_text(c, f"{livro}", width - 100, 50, height/2 - 20, "Helvetica", 16, 22, colors.dimgrey)
    draw_wrapped_text(c, capitulo, width - 100, 50, height/2 - 60, "Helvetica-Bold", 18, 26, colors.black)
    c.save()
    packet.seek(0)
    return PdfReader(packet).pages[0]

async def extract_toc_with_gemini(client, pdf_path: str) -> list:
    try:
        doc = fitz.open(pdf_path)
        toc_text = ""
        toc_pages = []
        max_scan = min(35, len(doc))
        for i in range(max_scan):
            page_text = doc[i].get_text("text")
            page_lower = page_text.lower()
            if any(kw in page_lower for kw in ["sumário", "sumario", "índice", "indice", "table of contents", "conteúdo", "conteudo"]):
                toc_pages.append(i)
        if toc_pages:
            selected_indices = set()
            for p in toc_pages:
                for offset in range(-1, 5):
                    idx = p + offset
                    if 0 <= idx < len(doc):
                        selected_indices.add(idx)
            for idx in sorted(selected_indices):
                toc_text += f"\n--- PÁGINA {idx+1} ---\n" + doc[idx].get_text("text")
        else:
            for i in range(min(25, len(doc))):
                toc_text += f"\n--- PÁGINA {i+1} ---\n" + doc[i].get_text("text")
        doc.close()
        if not toc_text.strip(): return []
            
        sys_prompt = """[O] - OBJETIVO
Atuar como um especialista em estruturação de metadados, extraindo o Sumário (Table of Contents - TOC) exato e completo de livros de medicina.

[C] - CONTEXTO
O texto fornecido representa as páginas iniciais do livro contendo o Sumário ou Índice. As quebras de página estão indicadas no texto.

[A] - AÇÕES
1. Varra o texto buscando a seção "Sumário" ou "Índice".
2. Extraia exaustivamente CADA capítulo e subseção listados no sumário.
3. Para cada item, determine o Nível Hierárquico (ex: Parte I = 1, Capítulo = 2, Subseção = 3).
4. Extraia o Título do item fielmente.
5. Extraia o número da PÁGINA correspondente (tente retornar a página real do PDF, considerando os deslocamentos do sumário).

[N] - NORMAS (CRÍTICO)
- POSITIVO (COBERTURA COMPLETA): É OBRIGATÓRIO extrair a lista COMPLETA de capítulos do sumário, do primeiro ao ÚLTIMO capítulo listado (ex: Capítulo 1 até Capítulo 30+). NUNCA interrompa a extração nos primeiros capítulos.
- NEGATIVO: NUNCA invente itens ou pule seções do sumário.
- POSITIVO: Siga estritamente o Schema JSON exigido (TocExtracted).

[E] - EXEMPLOS
Input: "...Parte I - Princípios. 1. Biologia Celular... 14. 13. Invasão Tumoral e Metástase... 245..."
Output:
Nivel: 1, Titulo: "Parte I - Princípios", Pagina: 14
Nivel: 2, Titulo: "13. Invasão Tumoral e Metástase", Pagina: 245

[S] - SAÍDA
Retorne OBRIGATORIAMENTE os dados estruturados conforme o JSON Schema exigido (TocExtracted)."""

        logging.info(f"Extraindo TOC de {os.path.basename(pdf_path)} via LLM...")
        prompt = f"TEXTO DO SUMÁRIO DO LIVRO:\n{toc_text}"
        data = await call_gemini_with_fallback(client, prompt, sys_prompt, TocExtracted, max_retries=4)
        
        toc = []
        if data and data.itens:
            for item in data.itens:
                toc.append([item.nivel, item.titulo, item.pagina])
        return toc
    except Exception as e:
        logging.error(f"Erro na extração de TOC via LLM para {pdf_path}: {e}")
        return []

async def get_pdfs_tocs(client, folder_path):
    tocs = {}
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            filepath = os.path.join(folder_path, filename)
            try:
                doc = fitz.open(filepath)
                toc = doc.get_toc()
                doc.close()
                if not toc or len(toc) < 15:
                    print(f"⚠️ O arquivo {filename} não possui um sumário digital válido. Extraindo via IA...")
                    toc = await extract_toc_with_gemini(client, filepath)
                tocs[filename] = toc
            except Exception as e:
                logging.error(f"Erro ao ler TOC de {filename}: {e}")
    return tocs

def merge_and_sort_cortes(cortes):
    if not cortes: return []
    from collections import defaultdict
    grouped = defaultdict(list)
    for c in cortes: grouped[c.arquivo].append(c)
    merged = []
    for arquivo, lista in grouped.items():
        lista.sort(key=lambda x: x.pagina_inicial)
        curr = lista[0]
        for nxt in lista[1:]:
            if nxt.pagina_inicial <= curr.pagina_final + 1:
                curr.pagina_final = max(curr.pagina_final, nxt.pagina_final)
            else:
                merged.append(curr)
                curr = nxt
        merged.append(curr)
    merged.sort(key=lambda x: {"conceito": 0, "mecanismo": 1, "clinica": 2}.get(x.nivel, 99))
    return merged

def agrupar_objetivos(objetivos):
    def get_pages(obj):
        pages = set()
        for c in obj.cortes:
            for p in range(c.pagina_inicial, max(c.pagina_inicial, c.pagina_final) + 1):
                pages.add(f"{c.arquivo}_{p}")
        return pages
    grupos = []
    for obj in objetivos:
        merged = False
        obj_pages = get_pages(obj)
        for grupo in grupos:
            grupo_pages = set()
            for g_obj in grupo:
                grupo_pages.update(get_pages(g_obj))
            if not obj_pages or not grupo_pages: continue
            overlap = len(obj_pages.intersection(grupo_pages)) / len(obj_pages.union(grupo_pages))
            if overlap > 0.3:
                grupo.append(obj)
                merged = True
                break
        if not merged:
            grupos.append([obj])
    objetivos_mesclados = []
    for grupo in grupos:
        if len(grupo) == 1:
            objetivos_mesclados.append(grupo[0])
        else:
            numeros = " e ".join([g.numero for g in grupo])
            titulo = "  |  ".join([f"[{g.numero}] {g.titulo}" for g in grupo])
            cortes = []
            videos = []
            for g in grupo:
                cortes.extend(g.cortes)
                if hasattr(g, 'videos') and g.videos:
                    videos.extend(g.videos)
            objetivos_mesclados.append(Objetivo(numero=numeros, titulo=titulo, cortes=cortes, videos=videos))
    return objetivos_mesclados

def calcular_pagina_final_inteligente(source_pdf, pag_ini, corte_pag_final):
    if corte_pag_final > pag_ini:
        return corte_pag_final
    try:
        doc = fitz.open(source_pdf)
        toc = doc.get_toc()
        doc.close()
        if toc:
            next_pages = sorted([item[2] for item in toc if isinstance(item[2], int) and item[2] > pag_ini])
            if next_pages:
                return max(pag_ini + 3, next_pages[0] - 1)
    except Exception:
        pass
    return pag_ini + 15

def gerar_pdfs(roteiro, pdfs_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    objetivos_processados = agrupar_objetivos(roteiro.objetivos)
    for obj in objetivos_processados:
        obj_pdf_path = os.path.join(output_dir, f"Objetivo {obj.numero}.pdf")
        writer = PdfWriter()
        writer.add_page(create_cover_page(obj.numero, obj.titulo, getattr(obj, 'videos', [])))
        cortes = merge_and_sort_cortes(obj.cortes)
        last_book_chapter = ""
        for corte in cortes:
            source_pdf = os.path.join(pdfs_dir, corte.arquivo)
            if not os.path.exists(source_pdf): continue
            
            # Ajuste Inteligente da Página Final baseado na estrutura do Sumário real do Livro
            if corte.pagina_final <= corte.pagina_inicial:
                corte_original_final = corte.pagina_final
                corte.pagina_final = calcular_pagina_final_inteligente(source_pdf, corte.pagina_inicial, corte.pagina_final)
                logging.info(f"ℹ️ Ajuste inteligente de capítulo em '{corte.arquivo}': {corte.pagina_inicial} -> {corte.pagina_final} (Calculado via limites do sumário).")
                
            curr_bc = f"{corte.arquivo}_{corte.capitulo}"
            if curr_bc != last_book_chapter:
                writer.add_page(create_separator_page(corte.arquivo, corte.capitulo))
                last_book_chapter = curr_bc
            reader = PdfReader(source_pdf)
            p_ini, p_fim = max(0, corte.pagina_inicial - 1), min(len(reader.pages), max(corte.pagina_inicial, corte.pagina_final))
            for p_num in range(p_ini, p_fim): writer.add_page(reader.pages[p_num])
        with open(obj_pdf_path, "wb") as f: writer.write(f)
        logging.info(f"Gerado: {obj_pdf_path}")

async def call_gemini_with_fallback(client, contents, system_instruction, response_schema, max_retries=4):
    models = ['gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-2.5-flash', 'gemini-3.5-flash']
    
    for model_to_use in models:
        for tentativa in range(max_retries):
            try:
                response = await client.aio.models.generate_content(
                    model=model_to_use,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=system_instruction,
                        response_mime_type="application/json",
                        response_schema=response_schema,
                    )
                )
                if not response.parsed:
                    raise ValueError("O modelo não retornou um JSON válido.")
                return response.parsed
            except Exception as e:
                error_msg = str(e)
                is_critical = any(term in error_msg for term in ['400', '401', 'InvalidArgument', 'PermissionDenied'])
                if is_critical:
                    logging.error(f"Erro fatal (Client Error): {e}. Abortando imediatamente (Fail-Fast).")
                    raise e
                if "404" in error_msg or "not found" in error_msg.lower() or "no longer available" in error_msg.lower():
                    logging.warning(f"Modelo {model_to_use} indisponível (Erro 404). Pulando para o próximo modelo...")
                    break
                exponent_cap = 2 ** tentativa
                sleep_time = random.uniform(0, exponent_cap)
                logging.error(f"Erro na API do Gemini com {model_to_use} (Tentativa {tentativa+1}/{max_retries}): {e}")
                logging.info(f"Aguardando {sleep_time:.2f}s antes da próxima tentativa...")
                await asyncio.sleep(sleep_time)
            
    return None

async def process_roteiro(objetivos_text, tocs, refs_dir):
    contexto = "SUMÁRIOS:\n"
    for arquivo, toc in tocs.items():
        if toc:
            contexto += f"Livro: {arquivo}\n"
            for item in toc:
                contexto += f"{'  '*(item[0]-1)}- {item[1]} (Página: {item[2]})\n"
    system_prompt = """[O] - OBJETIVO
Você é um orquestrador algorítmico de Roteiros de Tutoria (PBL) de Medicina. Sua função é mapear Objetivos de Aprendizagem para as subseções MAIS PROFUNDAS e ESPECÍFICAS possíveis dentro dos sumários de livros fornecidos, maximizando a precisão do corte.

[C] - CONTEXTO
Você recebe os sumários (TOC) extraídos dos livros em PDF (Ground Truth). Cada linha do sumário possui Nível Hierárquico, Título e Página Inicial. Você SÓ PODE usar páginas e títulos que existam no TOC fornecido.

[A] - AÇÕES
Pense passo a passo para cada objetivo:
1. Desconstrua os temas centrais do objetivo de aprendizagem.
2. Varra os sumários fornecidos buscando a subseção exata que responde a cada tema.
3. Se o objetivo contiver temas como 'metástase', 'invasão', 'angiogênese', busque a subseção exata correspondente no sumário (ex: 'Invasão Tumoral e Metástase'). NUNCA selecione 'Introdução' se houver um capítulo de metástase.
4. Defina a `pagina_inicial` correspondente à subseção mapeada.
5. Defina a `pagina_final` sendo exatamente a página do capítulo IMEDIATAMENTE SEGUINTE no sumário, menos 1. Se o Capítulo 13 é p.260 e o Capítulo 14 é p.275, a pagina_final de 13 DEVE ser 274.
6. Classifique o nível didático do corte (`conceito`, `mecanismo` ou `clinica`).

[N] - NORMAS (CRÍTICO)
- POSITIVO/NORMAS: A `pagina_final` DEVE abranger a totalidade da seção/capítulo (ex: se o capítulo vai de p.260 até p.274, pagina_final DEVE ser 274). NUNCA retorne pagina_final igual a pagina_inicial (ex: 260 -> 260).
- NEGATIVO: NUNCA mapeie um capítulo inteiro genérico (ex: "Introdução ao Câncer") se o objetivo versar sobre temas avançados (ex: "Metástase") e o sumário contiver o capítulo específico.
- NEGATIVO: NUNCA crie um corte gigante (ex: 30 páginas) para englobar temas distintos se houver "lixo" no meio. Crie objetos independentes.
- NEGATIVO: NUNCA invente páginas ou subseções que não estão no texto do sumário.
- POSITIVO: Busque cortes cirúrgicos abrangendo o capítulo útil completo.

[E] - EXEMPLOS (FEW-SHOT)
Exemplo 1 (Sucesso):
Objetivo: "Efeitos da Maconha e Cocaína".
Sumário Disponível:
Nível 1: Drogas
Nível 2: Cocaína (p.50)
Nível 2: Heroína (p.60)
Nível 2: Maconha (p.70)
Nível 2: Álcool (p.80)

Raciocínio: O objetivo pede Maconha e Cocaína. "Drogas" é muito amplo. "Heroína" e "Álcool" são ruídos. Devo separar os temas.
Cortes Gerados: 
- Corte 1: "Cocaína" (p.50 até 59).
- Corte 2: "Maconha" (p.70 até 79).

Exemplo 2 (Metástase):
Objetivo: "Fisiopatologia da metástase e cascata metastática".
Sumário Disponível:
Nível 1: 1 Introdução ao Câncer (p.1)
Nível 1: 13 Invasão Tumoral e Metástase (p.260)
Nível 1: 14 Diagnóstico das Neoplasias (p.275)

Raciocínio: O objetivo pede metástase. O capítulo 13 aborda "Invasão Tumoral e Metástase" (p.260). O capítulo 14 inicia em p.275. O corte deve ser de p.260 até p.274.
Cortes Gerados: 
- Corte 1: "13 Invasão Tumoral e Metástase" (p.260 até 274).

[S] - SAÍDA
Retorne OBRIGATORIAMENTE os dados estruturados conforme o JSON Schema exigido pela API (`RoteiroTutoria`). NÃO encapsule a resposta em blocos de código Markdown (como ```json). Retorne o JSON puro e direto."""
    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)
    parsed_response = await call_gemini_with_fallback(
        client=client,
        contents=f"{contexto}\n\nOBJETIVOS:\n{objetivos_text}",
        system_instruction=system_prompt,
        response_schema=RoteiroTutoria
    )
    return parsed_response

def extrair_texto_paginas(pdf_path, p_ini, p_fim):
    try:
        doc = fitz.open(pdf_path)
        text = ""
        start = max(0, p_ini - 1)
        end = min(len(doc), max(p_ini, p_fim))
        for i in range(start, end):
            text += f"\n--- PÁGINA {i+1} ---\n"
            text += doc[i].get_text("text")
        doc.close()
        return text
    except Exception as e:
        logging.error(f"Erro extraindo texto: {e}")
        return ""

async def refinar_roteiro(roteiro, refs_dir):
    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)
    sys_prompt = """[O] - OBJETIVO
Atue como um refinador algorítmico de cortes de leitura (PBL). Sua missão é extrair exatamente UM BLOCO ÚNICO CONTÍGUO de páginas que possua densidade informacional máxima para responder ao objetivo, minimizando o ruído.

[C] - CONTEXTO
Você recebe dois dados de entrada (Ground Truth):
1. O Objetivo de Aprendizagem.
2. O Texto Extraído do livro, onde cada transição de página é rigorosamente demarcada pelo marcador '--- PÁGINA X ---'. O texto das páginas reflete o layout real (com cabeçalhos, rodapés e quebras de linha).

[A] - AÇÕES
Pense passo a passo:
1. Desconstrua o objetivo do aluno para encontrar sua espinha dorsal semântica (o foco clínico, patológico ou farmacológico real).
2. Varra o Texto Extraído buscando a PRIMEIRA PÁGINA exata onde esse núcleo semântico começa a ser tecnicamente conceituado. Salve-a como `pagina_inicial_refinada`.
3. Avance a leitura a partir da página inicial cobrindo toda a explicação técnica do capítulo.
4. Marque `pagina_final_refinada` ao fim do tópico ou transição para assunto não correlato.

[N] - NORMAS (CRÍTICO)
- NEGATIVO: NUNCA crie um corte de apenas 1 página (ex: 260 -> 260) se o assunto for abrangente (ex: fisiopatologia e cascata de metástase). O corte DEVE conter o desenvolvimento técnico completo do capítulo (mínimo recomendável de 3 a 12 páginas).
- NEGATIVO: NUNCA crie múltiplos blocos. Retorne apenas UM bloco contíguo único para não quebrar a fluidez de leitura.
- NEGATIVO: NUNCA invente ou adivinhe números de página. Os números DEVEM ser estritamente os mesmos extraídos da tag '--- PÁGINA X ---'.
- POSITIVO: Seja cirúrgico sem mutilar o capítulo.

[E] - EXEMPLOS
Input (Simplificado):
Objetivo: "Diferenciar tolerância de abstinência."
Texto Extraído: 
--- PÁGINA 10 ---
Introdução. Histórico das drogas.
--- PÁGINA 11 ---
Tolerância: necessidade de doses maiores...
--- PÁGINA 12 ---
Síndrome de Abstinência: sinais e sintomas...
--- PÁGINA 13 ---
Tratamento e desintoxicação.

Chain of Thought & Output:
Raciocínio: A página 10 é gordura histórica. O núcleo inicia na 11 e se estende até a 12. A página 13 aborda tratamento, que é um novo assunto primário que foge de "diferenciar tolerância de abstinência". O corte deve ser 11 a 12.
Resultado: `relevante: true`, `pagina_inicial_refinada: 11`, `pagina_final_refinada: 12`.

[S] - SAÍDA
Retorne OBRIGATORIAMENTE os dados estruturados conforme o JSON Schema exigido pela API (`RefinamentoCorte`), sem explicações adicionais ou saudações."""
    for obj in roteiro.objetivos:
        novos_cortes = []
        for corte in obj.cortes:
            source_pdf = os.path.join(refs_dir, corte.arquivo)
            if not os.path.exists(source_pdf):
                novos_cortes.append(corte)
                continue
            texto = extrair_texto_paginas(source_pdf, corte.pagina_inicial, corte.pagina_final)
            if not texto.strip():
                novos_cortes.append(corte)
                continue
            prompt = f"OBJETIVO: {obj.titulo}\n\nTEXTO EXTRAÍDO:\n{texto[:150000]}"
            parsed_response = await call_gemini_with_fallback(
                client=client,
                contents=prompt,
                system_instruction=sys_prompt,
                response_schema=RefinamentoCorte
            )
            if parsed_response:
                if parsed_response.relevante:
                    corte.pagina_inicial = parsed_response.pagina_inicial_refinada
                    corte.pagina_final = parsed_response.pagina_final_refinada
                    novos_cortes.append(corte)
            else:
                novos_cortes.append(corte)
        if novos_cortes:
            obj.cortes = novos_cortes
        else:
            if len(obj.cortes) > 0:
                logging.warning(f"O Agente 2 rejeitou TODOS os cortes para o objetivo {obj.numero}. Mantendo os originais.")
            else:
                logging.warning(f"O Agente 1 NÃO ENCONTROU nenhum capítulo no sumário para o objetivo {obj.numero}.")
    return roteiro

async def planejar_videos(client, objetivo_titulo):
    sys_prompt = """**OBJETIVO:**
Atuar como um Planejador Pedagógico Clínico. Sua missão é ler um Objetivo de Aprendizagem e fragmentá-lo em termos de busca (queries) estritos, que serão utilizados para encontrar videoaulas essenciais no YouTube em Português.

**CONTEXTO:**
Você receberá o enunciado completo de um Objetivo de Aprendizagem de um Roteiro de Tutoria (PBL) de Medicina.

**AÇÕES:**
1. Desconstrua o objetivo para identificar seus eixos principais (ex: Fisiopatologia, Diagnóstico, Farmacologia, Epidemiologia).
2. Avalie a necessidade real de suporte visual ou em vídeo para cada eixo.
3. Se o objetivo contiver múltiplos agentes, patologias ou drogas (ex: Cocaína, Maconha, Álcool), fragmente a pesquisa gerando um termo de busca separado para cada entidade.
4. Para cada eixo/entidade relevante, gere um termo de busca clínico e direto em Português (ex: "Intoxicação por Cocaína Fisiopatologia").

**NORMAS:**
1. **Contenção Trivial:** É terminantemente PROIBIDO recomendar vídeos para objetivos puramente epidemiológicos, históricos, ou sociológicos.
2. **Formatação de Query:** NUNCA inclua as palavras "medicina" ou "aula" nos termos gerados (o sistema injetará isso automaticamente no backend).
3. **Limite de Fragmentação:** Não gere mais do que 4 termos por objetivo.
4. **Formato de Saída:** Retorne estritamente o objeto JSON conforme o schema exigido (PlanejamentoVideos).

**SAÍDA:**
Retornar objeto JSON aderente ao Schema PlanejamentoVideos."""
    return await call_gemini_with_fallback(client, f"OBJETIVO:\n{objetivo_titulo}", sys_prompt, PlanejamentoVideos)

ENGLISH_TERMS = ['pathology', 'surgery', 'lecture', 'overview', 'treatment of', 'management of', 'diagnosis of', 'journal', 'usmle', 'role in', 'review of', 'case report', 'clinical trial', 'definition', 'understanding', 'mechanism of', 'syndrome']

def eh_titulo_em_ingles(titulo):
    t_lower = titulo.lower()
    indicadores_pt = ['aula', 'medicina', 'resumo', 'fisiopatologia', 'tratamento', 'diagnostico', 'diagnóstico', 'doença', 'síndrome', 'sindrome', 'sanar', 'jaleko', 'estrategia', 'estratégia', 'medway', 'afya', 'medcel']
    if any(pt in t_lower for pt in indicadores_pt):
        return False
    return any(eng in t_lower for eng in ENGLISH_TERMS)

async def buscar_youtube(termo, api_key):
    query = urllib.parse.quote(f"{termo} medicina aula")
    url = f"https://www.googleapis.com/youtube/v3/search?part=snippet&q={query}&type=video&maxResults=5&key={api_key}&relevanceLanguage=pt"
    async with httpx.AsyncClient() as client:
        try:
            r = await client.get(url)
            if r.status_code == 200:
                data = r.json()
                resultados = []
                termo_proibidos = ["música", "musica", "clipe", "official video", "video oficial", "karaoke", "paródia", "parodia", "rick astley"]
                for item in data.get('items', []):
                    snippet = item.get('snippet', {})
                    title = snippet.get('title', '')
                    t_lower = title.lower()
                    if any(p in t_lower for p in termo_proibidos) or eh_titulo_em_ingles(title):
                        logging.info(f"Desconsiderando vídeo inadequado/em inglês no backend: '{title}'")
                        continue
                    resultados.append({
                        'id': item['id']['videoId'],
                        'title': title,
                        'channel': snippet.get('channelTitle', ''),
                        'description': snippet.get('description', '')
                    })
                return resultados
        except Exception as e:
            logging.error(f"Erro no YouTube API: {e}")
    return []

async def avaliar_com_llm(client, termo, resultados):
    if not resultados: return None
    resultados_txt = ""
    for i, vid in enumerate(resultados):
        resultados_txt += f"\nOpção {i+1}:\n- ID: {vid['id']}\n- Título: {vid['title']}\n- Canal: {vid['channel']}\n- Descrição: {vid['description']}\n"
    sys_prompt = """**OBJETIVO:**
Atuar como Curador Acadêmico Médico rigoroso. Sua missão é analisar uma lista de resultados de busca do YouTube e selecionar O MELHOR material (videoaula) para estudantes de medicina e residentes.

**CONTEXTO:**
Você receberá o TEMA / TERMO DE BUSCA, e as opções retornadas pela API do YouTube, contendo Título, Canal e Descrição.

**AÇÕES:**
1. Analise o Tema/Termo para entender o foco clínico exigido.
2. Varra os metadados (Título, Canal, Descrição) de cada vídeo disponível.
3. Classifique a Autoridade do Canal: priorize canais de educação médica consolidados (ex: SanarFlix, Estratégia MED, Medway, Afya, Medcel, Ligas Acadêmicas).
4. Eleja a opção de maior profundidade científica.
5. Se não houver nenhum candidato aceitável ou em português, defina o ID escolhido como 'NENHUM' e o título formatado vazio.

**NORMAS:**
1. **Filtro de Leigos:** REJEITE sumariamente vídeos direcionados a pacientes leigos (ex: "quais os sintomas", "como curar", "o que é", "dicas de saúde").
2. **Filtro de Idioma (ESTRITO):** É TERMINANTEMENTE PROIBIDO selecionar vídeos em inglês, espanhol ou qualquer idioma estrangeiro. A videoaula DEVE ser estritamente em Português do Brasil (PT-BR). Se todas as opções forem em idioma estrangeiro, defina o `video_escolhido_id` obrigatoriamente como 'NENHUM'.
3. **Alucinação Zero:** NUNCA invente um ID de vídeo que não esteja explicitamente listado nas opções. O ID retornado DEVE ser uma das opções fornecidas.
4. **Formato de Saída:** Retorne estritamente o objeto JSON conforme o schema exigido (CuradoriaVideo).

**SAÍDA:**
Retornar objeto JSON aderente ao Schema CuradoriaVideo."""
    prompt = f"Tema/Termo: {termo}\n\nOpções:\n{resultados_txt}"
    return await call_gemini_with_fallback(client, prompt, sys_prompt, CuradoriaVideo)

async def adicionar_videos_ao_roteiro(roteiro):
    api_key_youtube = os.environ.get('YOUTUBE_API_KEY')
    api_key_gemini = os.environ.get('GEMINI_API_KEY')
    if not api_key_youtube:
        logging.warning("YOUTUBE_API_KEY não encontrada. Pulando curadoria de vídeos.")
        return roteiro
    client = genai.Client(api_key=api_key_gemini)
    print("\nIniciando Agente 3: Curadoria de Vídeos do YouTube...")
    for obj in roteiro.objetivos:
        print(f"  -> Analisando vídeos para: Objetivo {obj.numero}")
        plan = await planejar_videos(client, obj.titulo)
        if plan and plan.termos_busca:
            print(f"     Termos identificados: {plan.termos_busca}")
            for termo in plan.termos_busca:
                resultados = await buscar_youtube(termo, api_key_youtube)
                valid_ids = {vid['id']: vid['title'] for vid in resultados}
                curadoria = await avaliar_com_llm(client, termo, resultados)
                if curadoria and curadoria.video_escolhido_id in valid_ids and curadoria.video_escolhido_id != "NENHUM":
                    if not hasattr(obj, 'videos') or obj.videos is None:
                        obj.videos = []
                    final_title = curadoria.titulo_formatado or valid_ids[curadoria.video_escolhido_id]
                    obj.videos.append(VideoSugerido(
                        termo_busca=termo,
                        video_id=curadoria.video_escolhido_id,
                        titulo_formatado=final_title
                    ))
                    print(f"     ✅ Vídeo Curado para '{termo}': {final_title}")
                else:
                    print(f"     ⚠️ Nenhum vídeo bom em PT-BR encontrado para '{termo}'.")
    return roteiro

async def gerar_esboco(objetivos_text, refs_dir):
    api_key = os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=api_key)
    print("Analisando sumários e consultando a API do Gemini (Agente 1)...")
    tocs = await get_pdfs_tocs(client, refs_dir)
    if not tocs:
        print(f"❌ Erro crítico: Nenhum arquivo PDF com sumário foi encontrado na pasta '{refs_dir}'.")
        print("Verifique se o caminho da variável PASTA_LIVROS está correto e se há arquivos .pdf dentro dela.")
        return
    roteiro = await process_roteiro(objetivos_text, tocs, refs_dir)
    if roteiro:
        print("Refinando cortes através de leitura de texto (Agente 2)...")
        roteiro = await refinar_roteiro(roteiro, refs_dir)
        roteiro = await adicionar_videos_ao_roteiro(roteiro)
        with open('roteiro_revisao.json', 'w', encoding='utf-8') as f:
            f.write(roteiro.model_dump_json(indent=4))
        print("✅ Esboço gerado com sucesso!")
        print("👉 Dê um clique duplo no arquivo 'roteiro_revisao.json' no menu de Arquivos à esquerda.")
        print("👉 Revise as páginas e os cortes. Se algo estiver errado, altere os números e salve (Ctrl+S).")
        print("👉 Depois de salvar, rode a célula '6. EXPORTAR PDFs FINAIS'.")
    else:
        print("❌ Falha ao gerar o roteiro.")


In [ ]:
# 5. INSERIR OBJETIVOS E GERAR ESBOÇO
# ===================================
PASTA_LIVROS = "/content/drive/MyDrive/Logística - Drive/Tutoria"
PASTA_SAIDA = "/content/drive/MyDrive/Logística - Drive/Tutoria/saida/Tutoria_Teste"

OBJETIVOS = """
1. Descrever as drogas ilícitas mais comuns...
2. Confrontar os conceitos clínicos e neurobiológicos...
"""

await gerar_esboco(OBJETIVOS, PASTA_LIVROS)


In [ ]:
# 6. EXPORTAR PDFs FINAIS (Após sua revisão!)
# ==========================================
import json
import os
import logging

def exportar_pdfs_finais():
    print("Lendo roteiro_revisao.json...")
    if not os.path.exists('roteiro_revisao.json'):
        print("❌ Erro: Arquivo roteiro_revisao.json não encontrado. Rode a célula 5 primeiro.")
        return
        
    with open('roteiro_revisao.json', 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
    # Normalização defensiva de chaves (Self-Healing contra erros de digitação como 'titux' ou 'título')
    if isinstance(dados, dict) and "objetivos" in dados:
        for obj in dados.get("objetivos", []):
            if "titux" in obj and "titulo" not in obj:
                print("⚠️ Aviso: Chave incorreta 'titux' detectada em objetivo. Corrigindo automaticamente para 'titulo'...")
                obj["titulo"] = obj.pop("titux")
            if "título" in obj and "titulo" not in obj:
                obj["titulo"] = obj.pop("título")
            if "title" in obj and "titulo" not in obj:
                obj["titulo"] = obj.pop("title")
            if "cortes" in obj:
                for corte in obj.get("cortes", []):
                    if "título" in corte and "capitulo" not in corte:
                        corte["capitulo"] = corte.pop("título")
                    if "capítulo" in corte and "capitulo" not in corte:
                        corte["capitulo"] = corte.pop("capítulo")

    roteiro = RoteiroTutoria.model_validate(dados)
    print("Roteiro carregado com sucesso. Agrupando e gerando PDFs...")
    gerar_pdfs(roteiro, PASTA_LIVROS, PASTA_SAIDA)
    print("✅ Todos os PDFs foram gerados e salvos no Drive!")

exportar_pdfs_finais()
